# Configure the S04S10 postfire ATS–Lambda–PFLOTRAN case

Run from the **case's `notebooks/` directory**, using the existing watershed-workflow Python environment.
This notebook reviews fire forcing, clones both prefire reactive XML variants, copies the parent reactions,
and writes `caselambda-run2-postfire.s1`. It does not submit or execute ATS.

The default is the realistic **P_map**, low severity. MODIS LAI and ELM DOC already include fire effects;
no additional LAI or DOC multiplier is applied. Vegetation curves describe NLCD classes, not individual species.

In [ ]:
from pathlib import Path
import json
import pickle
import h5py
import numpy as np
import pandas as pd
from scipy.io import loadmat
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from rasterio.plot import show
from IPython.display import display
from config_utils import load_fire_workflow, first_postfire_year, load_config, phase_dates, full_timeline_forcing_dir
from postfire_utils import (SEVERITY, CLASS_LABELS, DOC_KEY, mesh_first_layer,
                           burn_review, validate_forcing, generate_case, require)

notebooks_dir = Path.cwd().resolve()
require(notebooks_dir.name == 'notebooks' and (notebooks_dir / 'config.json').is_file(),
        'Run from the case notebooks/ directory.')
base_dir = notebooks_dir.parent
config = load_config(notebooks_dir / 'config.json')
fire = load_fire_workflow(config, base_dir, artificial=False)
site_name = config['case']['site_name']
nx = config['case']['meshsize_nx']

# Editable user choices; inspect the burn map below before generating the case.
flag_scenario = fire['scenario']
burn_severity_class = fire['severity']  # unburned/no-low, low, moderate, high

burn_raster = fire['burn_raster']
site_dir = base_dir / 'data-processed' / site_name
mat_path = site_dir / f'm2_coords_{site_name}.mat'
layer_data = loadmat(mat_path)
nz = layer_data['dzs_soil'].size + layer_data['dzs_geo'].size
mesh_path = site_dir / f'2dtransect_{site_name}_nx{nx}_nz{nz}.exo'
mesh_info = mesh_first_layer(mesh_path, mat_path, nx)
nz = mesh_info['nz']
parent_xml = base_dir / f'caselambda-run2-prefire.{flag_scenario}' / f'{site_name}_nx{nx}_nz{nz}.run2.v1.6_pflotran.xml'
forcing_dir = full_timeline_forcing_dir(config, base_dir / 'data-processed')
pre_dates = phase_dates(config, 'prefire_transient')
post_dates = phase_dates(config, 'postfire_transient')
start_day = config['spinup']['cyclic_years'] * 365 + len(pre_dates)
end_day = start_day + len(post_dates)
review_figures = {}
print(f'{site_name}, nx={nx}, nz={nz}; postfire {post_dates[0]} through {post_dates[-1]}')
print(f'ATS no-leap clock: {start_day} to {end_day} d; {len(post_dates)} postfire days')

## 1. Review the burn-severity map and select the class

The map uses the original categorical 30 m MTBS product. The line is the actual shortened hillslope,
not the raw candidate transect or its receiving river segment. Fractions are weighted by the line's
intersection length with each raster pixel; non-processing and increased-greenness codes are excluded.
Ties select the higher class. The editable selection above controls permeability; it does not alter LAI/DOC.

In [ ]:
# The site-selection cache stores the CRS used by the saved start/end coordinates.
with (notebooks_dir / 'site_selection/river_cells_and_segments.pkl').open('rb') as handle:
    transect_crs = pickle.load(handle)['crs']
burn_data, burn_transform, transect, burn_report = burn_review(
    burn_raster, site_dir / f'startendcoords_{site_name}_summary.csv', transect_crs)
colors = ['white', 'darkgreen', 'aquamarine', 'gold', 'red', 'limegreen']
cmap = ListedColormap(colors)
norm = BoundaryNorm(np.arange(-0.5, 6.5), cmap.N)
fig, ax = plt.subplots(figsize=(10, 7))
show(burn_data, transform=burn_transform, ax=ax, cmap=cmap, norm=norm)
xy = np.array(transect.coords)
ax.plot(xy[:, 0], xy[:, 1], color='black', lw=2, label=f'Shortened {site_name} transect')
ax.scatter(*xy[0], marker='^', s=90, color='blue', label='Upslope start')
ax.scatter(*xy[-1], marker='o', s=70, color='blue', label='Toe endpoint')
colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=range(6))
colorbar.ax.set_yticklabels([CLASS_LABELS[i] for i in range(6)])
ax.set(title=f'{site_name}: burn severity and actual transect', xlabel='Easting [m]', ylabel='Northing [m]')
ax.legend(); ax.set_aspect('equal'); plt.tight_layout()
review_figures['burn_severity'] = fig
plt.show()
fractions = pd.DataFrame([{'class': CLASS_LABELS[int(k)], 'length_fraction': v}
                          for k, v in burn_report['class_fractions'].items()])
display(fractions)
require(burn_severity_class in SEVERITY, f'Choose one of {list(SEVERITY)}')
selected_code, multiplier = SEVERITY[burn_severity_class]
print(f'Map dominant class: {CLASS_LABELS[burn_report["dominant_class"]]}; selected: {burn_severity_class}; factor: {multiplier}')
print(f'Valid classified length: {burn_report["valid_length_m"]:.1f} / {burn_report["transect_length_m"]:.1f} m; tie: {burn_report["tie"]}')
if selected_code != burn_report['dominant_class']:
    print('Selection differs from the map-derived dominant class. The user selection remains authoritative.')

## 2. Review the prefire and postfire LAI and DOC forcing

Read the canonical full timeline used by the model. Convert no-leap days using `phase_dates`, so
leap years do not shift the displayed dates. The fire marker is the configured postfire start.
DOC is a spatial mean of the existing volumetric injection field, not a hillslope-integrated mass rate.

In [ ]:
plot_dates = pd.to_datetime(pre_dates + post_dates)
spinup_days = config['spinup']['cyclic_years'] * 365
selection = slice(spinup_days, end_day)
with h5py.File(forcing_dir / f'{site_name}_MODIS_LAI.h5') as hdf:
    lai_keys = [k for k in hdf if k.endswith(' LAI [-]')]
    require(len(lai_keys) > 0, 'No vegetation LAI curves found')
    fig, axes = plt.subplots(len(lai_keys), 1, figsize=(12, 3 * len(lai_keys)), sharex=True, squeeze=False)
    for ax, key in zip(axes[:, 0], lai_keys):
        ax.plot(plot_dates, hdf[key][selection], lw=1, label=key)
        ax.axvline(pd.Timestamp(post_dates[0]), color='firebrick', ls='--', label='Postfire start')
        ax.set(ylabel='LAI [-]', title=key.replace('NLCD ', '').replace(' LAI [-]', ''))
        ax.grid(alpha=0.25); ax.legend()
    axes[-1, 0].set_xlabel('Calendar date')
    plt.tight_layout()
    review_figures['lai_prefire_postfire'] = fig
    plt.show()
with h5py.File(forcing_dir / f'{site_name}_DOC_source.h5') as hdf:
    source_mean = np.array([hdf[DOC_KEY][str(i)][:].mean() for i in range(spinup_days, end_day)])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_dates, source_mean, color='tab:green', lw=0.9)
ax.axvline(pd.Timestamp(post_dates[0]), color='firebrick', ls='--', label='Postfire start')
ax.set(xlabel='Calendar date', ylabel='Mean DOC injection [molC m$^{-3}$ s$^{-1}$]', title='Existing prefire and postfire DOC injection')
ax.legend(); ax.grid(alpha=0.25); plt.tight_layout()
review_figures['doc_prefire_postfire'] = fig
plt.show()
post = config['postfire_transient']
print('ELM ignition-day realization:', post.get('ignition_day_elm_run', post['elm_run']))
print('ELM subsequent postfire realization:', post['elm_run'])

## 3. First subsurface layer and Li permeability rule

The affected horizon is **layer 1 of the ATS subsurface domain**, exactly 0.05 m thick.
The mesh surface side set identifies its adjacent cells; NFACED connectivity verifies their vertical
thickness. These cells are not consecutive Exodus IDs because the mesh is grouped by material.

Following the project's close reading of [Li et al. (2026), Eqs. 4–5](https://doi.org/10.1016/j.jhydrol.2025.134538):
unburned/no-low and low retain permeability; moderate uses 0.40 times and high 0.20 times baseline.
Only the first layer changes, preserving each material's baseline value and all deeper values.
In the reference nx50 mesh, 29 first-layer cells use GLHYMPS material 100 and 21 use NRCS materials.
The factor applies to all nx first-layer cells, preserving their existing material assignments.

**Mesh-attached regions for every severity class:** Generation copies the original mesh into the postfire case
and adds the element set `postfire-first-layer` (label 2000) using the nx first-layer Exodus element
indices. Original coordinates, connectivity, ordering, material blocks, and existing sets remain unchanged.
The copy is checked against every original mesh variable, dimension, and attribute.

Each affected NRCS or GLHYMPS material gets two logical XML regions: `xx-burned` is its intersection
with the first-layer set; `xx-deeper` is the original region minus that set. Permeability functions
use these disjoint subsets. The original material regions remain available to all ten DOC source
bins, porosity, compressibility, and water-retention properties. These source/property definitions
are preserved. Materials without first-layer cells keep their original permeability function.

This uses mesh labels carried through MPI partitioning and requires no runtime-ID CSV.
Low/no-low also create the copied mesh and burned/deeper regions, with permeability factor 1.0
on both subsets. This makes low severity a test of the same mesh/region/restart structure used
for moderate/high, without changing the permeability field. Use the same MPI rank count and
partition settings as prefire for a restart; mesh-content checks do not establish restart compatibility.


In [ ]:
print(f'First-layer thickness: {mesh_info["thickness_m"]:.3f} m')
print(f'First-layer cell count: {len(mesh_info["top"])} (expected {nx})')
first_layer_table = pd.DataFrame({
    'exodus_cell_1based': mesh_info['top'] + 1,
    'material_id': mesh_info['materials'][mesh_info['top']],
    'x': mesh_info['centroids'][mesh_info['top'], 0],
    'y': mesh_info['centroids'][mesh_info['top'], 1],
    'z': mesh_info['centroids'][mesh_info['top'], 2],
})
display(first_layer_table)
display(pd.DataFrame([{'class': name, 'BARC_index': code, 'first_layer_multiplier': factor}
                      for name, (code, factor) in SEVERITY.items()]))

## 4. Validate the canonical forcing and generate the postfire case

The checks scan every stored numeric dataset, validate the continuous daily clock and referenced
XML headers, and reconstruct total carbon from all ten donor species for sources and boundaries.
The full timeline's last sample is day `end_day - 1`. ATS tabular and bilinear/time functions hold
that last value through `end_day`, preserving the existing inclusive-day convention without
changing any forcing files. These checks may take a few minutes on the shared filesystem.

Generation always starts from the prefire parent, so rerunning cannot compound scaling or append
duplicate regions. Both XML variants are copied; the mass-flux observations and reaction files
remain identical. The target site directory is the intended ATS execution directory.

In [ ]:
print('Validating forcing clocks, values, headers, and donor carbon ...', flush=True)
forcing_report = validate_forcing(config, base_dir, parent_xml)
print('Forcing checks passed. Generating postfire XML variants ...', flush=True)
target_dir, report = generate_case(
    base_dir, config, flag_scenario, burn_severity_class, mesh_info, forcing_report,
    burn_report)
figures_dir = target_dir / 'figures'
figures_dir.mkdir(exist_ok=True)
for name, figure in review_figures.items():
    figure.savefig(figures_dir / f'{name}.png', dpi=150, bbox_inches='tight')
first_layer_table.to_csv(target_dir / 'first_layer_cells.csv', index=False)
display(pd.DataFrame(report['permeability']))
print('Generated:', target_dir)
print('Validation report:', target_dir / 'postfire_configuration_report.json')
print('Restart:', report['restart_path'])
print('Readiness:', report['readiness'])
print('Mesh:', report['mesh']['copy'] or report['mesh']['original'])
if report['mesh']['element_set_name']:
    print('First-layer set:', report['mesh']['element_set_name'], 'label', report['mesh']['element_set_label'])
for item in report['xml']:
    print(Path(item['output']).name, ':', len(item['changes']), 'reviewed XML changes')

## 5. Runtime acceptance after the prefire checkpoint is available

The notebook can generate configuration before `checkpoint_final.h5` exists. If it exists,
its ATS time attribute must equal the configured postfire start day (5052 for the reference case). Configuration validity
is distinct from runtime acceptance; the JSON report leaves `runtime_ready` false until a short
restart is checked. No full simulation is submitted here.

Run a separate short restart with the same ATS/PFLOTRAN build, MPI rank count, and partitioning
as prefire. At initialization and the first output, check continuity of pressure, saturation,
canopy/snow storage, and all transported/reactive concentrations; confirm postfire LAI, DOC,
meteorology, and both boundary heads use the configured postfire start day. Inspect initialized intrinsic permeability:
low must match prefire everywhere, and moderate/high must match the selected factor in exactly
the configured nx first-layer cells, with deeper cells unchanged. Independent permeability evaluators in
the inspected ATS source recompute on their first update, even when constant in time; verify
this behavior with the build actually used to run the case. Review water balance for the short run.

Review `postfire_configuration_report.json` for the original/copied mesh hashes, element-set membership,
logical region definitions, parent/output hashes, and exact XML differences,
reaction hashes, ELM provenance, severity selection, and restart status before the full run.